Processing data step
- connect data
- filter data  (remove unnecessary columns, remove blank rows, filter rows)
- clean data 
1. remove duplicates (use group by in pk to know duplicate situation)
2.  text: trim, case, replace
3. numbers: format, round, type
4. dates: format, type
5. handle missing values
- transform data (split columns, merge columns, extract parts)
- combine data (merge or append tables)
- clean up (remove unnecessary columns, rename table & columns "Naming Conventions", reorder the columns in logical order


In [100]:
import sys
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent          
sys.path.append(str(ROOT))

from config import RAW
from config import PROCESSED

PROCESSED.mkdir(parents=True, exist_ok=True)

### Connect The Data

In [101]:
# Youtube data
files = sorted((RAW / "youtube").glob("*_youtube.csv"))


df_list = []
for file in files:
    df = pd.read_csv(file)
    df["file_source"] = file.name
    df_list.append(df)

df_youtube = pd.concat(df_list, ignore_index=True)

# Releases data
df_releases = pd.read_csv(RAW / "releases.csv", parse_dates=["release_date"])

print(df_youtube.head())
print(df_releases.head())

      video_id                                              title  \
0  6C3hTze7h-M  #NINGNING at #ALO #aespa #æspa #에스파 #NINGNING ...   
1  ttjyzsQIWgQ  [ae:scenes] 애리, 가재 그리고 물고기들 🦞🐟 | 에스파 지젤 마리끌레르 ...   
2  JjceGR5Uvxw  walk it like i talk it #aespa #æspa #에스파 #GISE...   
3  nQGWYTN8Xrs  #GISELLE at #LOEWE #aespa #æspa #에스파 #GISELLE #지젤   
4  Qnv4UrQKrZE  happy 6 years anniversary to us 🎂 #aespa #æspa...   

                                         description       tags  \
0                                                NaN        NaN   
1  aespa Official\nhttps://www.youtube.com/c/aesp...  aespa|에스파   
2                                                NaN        NaN   
3                                                NaN        NaN   
4                                                NaN        NaN   

           published_at duration   views  likes  comments  \
0  2026-11-21T09:01:39Z    PT11S   76258   9060     126.0   
1  2026-11-20T13:00:33Z     PT7M   77447   2952     143.0   

In [102]:
yt_groups = set(df_youtube['group'].unique())
rel_groups = set(df_releases['group'].unique())

print("Group in YouTube :", sorted(yt_groups))
print("Group in Releases:", sorted(rel_groups))

# Check for differences in the way things are written between the two.
print("\nAvailable on YouTube but NOT suitable for Releases:", yt_groups - rel_groups)
print("Available on Releases but NOT suitable for Youtube:", rel_groups - yt_groups)


Group in YouTube : ['aespa', 'hearts2hearts', 'nct 127', 'nct dream', 'nct wish', 'riize', 'wayv']
Group in Releases: ['aespa', 'hearts2hearts', 'nct 127', 'nct dream', 'nct wish', 'riize', 'wayv']

Available on YouTube but NOT suitable for Releases: set()
Available on Releases but NOT suitable for Youtube: set()


### Cleaning Youtube Data
* **Duplicate Check:** Verified zero duplicate records (`0` duplicates found).
* **Missing Values:** Flagged disabled comments (`comments_disabled`) and imputed missing comment counts with `0`.
* **Data Standardization:** Converted `duration` to seconds (`int64`) and standardized `published_at` to timezone-naive `datetime`.
* **Feature Engineering:** Extracted `hashtags` from titles, stripped hashtags from `title`, and classified videos into `Shorts` vs. `Long`
* **Export:** Saved the clean dataset to `processed/youtube_clean.csv`.


In [103]:
df_clean = df_youtube.copy()

# print(df_clean.info())
# print(df_clean.columns.tolist())

# Check Duplicate Data
# print(df_clean.duplicated().sum()) (Duplicate = 0)

# Handle Missing Values
# print(df_clean.isnull().sum())
df_clean['comments_disabled'] = df_clean['comments'].isnull()
df_clean['comments'] = df_clean['comments'].fillna(0)
df_clean['comments'] = df_clean['comments'].astype('int64') # Change into Int type

# Clean the Format of Duration Columns
df_clean['duration'] = pd.to_timedelta(df_clean['duration']).dt.total_seconds().astype('int64') 

# Clean the Format of Published Colums
df_clean["published_at"] = pd.to_datetime(df_clean["published_at"]).dt.tz_localize(None)

# Extract Hashtags & Clean Title 
df_clean['hashtags'] = df_clean['title'].fillna('').apply(
    lambda x: ', '.join(re.findall(r'#\w+', x))
)
df_clean['title'] = df_clean['title'].fillna('').apply(
    lambda x: re.sub(r'#\w+', '', x).strip()
)

# Classify Video Type (Shorts vs Long) 
has_shorts_tag = (
    df_clean['hashtags'].str.contains('shorts', case=False, na=False) | 
    df_clean['title'].str.contains('shorts', case=False, na=False)
)

# Classify with Duration <= 180 is Shorts and The Title Contain Word or Hashtag Shorts
df_clean['video_type'] = np.where((df_clean['duration'] <= 180) | has_shorts_tag, 'Shorts', 'Long')
# print(df_youtube['video_type'].value_counts())

# Remove Unnecessary Columns 
columns_to_filter = ['video_id', 'description', 'tags', 'collected_at', 'file_source']
df_clean = df_clean.drop(columns=columns_to_filter, errors='ignore')

# Order Columns
column_order = [
    'group', 
    'video_type', 
    'title', 
    'hashtags', 
    'published_at', 
    'duration', 
    'views', 
    'likes', 
    'comments', 
    'comments_disabled'
]

df_clean = df_clean[column_order]
#print(df_clean)

# Save to Processed Data
df_clean.to_csv(PROCESSED / "youtube_clean.csv", index=False, encoding="utf-8-sig")

### Cleaning Releases Data
Performed a quick data quality audit on `df_releases` before exporting:
* **Data Integrity:** Verified schema/types via `.info()`, confirming **0 duplicates**, **0 missing values**, and **0 leading/trailing whitespaces**.
* **Export:** Saved the clean dataset to `processed/releases_clean.csv`


In [104]:
df_copy = df_releases.copy()
print(df_copy.info())

# Check for Duplicates and Missing Values
print("\nDuplicates:", df_copy.duplicated().sum())
print("\nMissing values:\n", df_copy.isnull().sum())

# Check Whitespaces
print("\nCheck Whitespaces")
for col in df_copy.select_dtypes(include=['object', 'string']).columns:
    whitespace = df_copy[col].dropna().str.contains(r'^\s+|\s+$', regex=True).sum()
    print(f"{col}: {whitespace} whitespace")

# Save to Proccessed Data
df_copy.to_csv(PROCESSED / "releases_clean.csv", index=False, encoding="utf-8-sig")


<class 'pandas.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   release_id        66 non-null     int64         
 1   group             66 non-null     str           
 2   release_name      66 non-null     str           
 3   release_date      66 non-null     datetime64[us]
 4   type              66 non-null     str           
 5   is_main_realeses  66 non-null     bool          
dtypes: bool(1), datetime64[us](1), int64(1), str(3)
memory usage: 2.8 KB
None

Duplicates: 0

Missing values:
 release_id          0
group               0
release_name        0
release_date        0
type                0
is_main_realeses    0
dtype: int64

Check Whitespaces
group: 0 whitespace
release_name: 0 whitespace
type: 0 whitespace
